In [7]:
import os
import uuid

from langchain_groq import ChatGroq
llm = ChatGroq(
    temperature=0,
    groq_api_key=os.getenv("GROQ_API_KEY"),
    model_name='llama3-70b-8192'
)

response = llm.invoke("first person to land on moon was....")
print(response.content)

That's an easy one!

The first person to set foot on the Moon was Neil Armstrong. He stepped out of the lunar module Eagle and onto the Moon's surface on July 20, 1969, during the Apollo 11 mission.

Armstrong famously declared, "That's one small step for man, one giant leap for mankind," as he became the first human to set foot on another celestial body.

He was followed by fellow astronaut Edwin "Buzz" Aldrin, who also walked on the Moon during the mission.


In [38]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://www.google.com/about/careers/applications/jobs/results/110690555461018310-software-engineer-iii-infrastructure-core")
page_data = loader.load().pop().page_content
print(page_data)

Software Engineer III, Infrastructure, Core — Google CareersCareersSkip navigation linkshomehomeHomeHomework_outlinework_outlineJobsJobsnoogler_hatnoogler_hatStudentsStudentsgooglegoogleHow we workHow we workhandymanhandymanHow we hireHow we hireperson_outlineperson_outlineYour careerYour careerhelp_outlineHelp linkfeedbackSend feedbackmore_vert HelpSend FeedbackSign inCareershomeHomework_outlineJobsexpand_morenoogler_hatStudentsexpand_moregoogleHow we workexpand_morehandymanHow we hireexpand_moreperson_outlineYour careerexpand_morejob detailsarrow_backBack to jobs searchJobs search results2,915  jobs matchedSoftware Engineer III, Infrastructure, CoreBengaluru, Karnataka, IndiaPersistent Disk Capacity Team LeaderKirkland, WA, USASenior Software Engineer, Android PartnerWarsaw, PolandSoftware Engineer III, Google CloudBengaluru, Karnataka, India; Pune, Maharashtra, India; +2 more; +1 moreAccount Manager, Large Customer Sales (English, Thai)Bangkok, ThailandGrowth Manager, App Developer 

In [22]:
from langchain_core.prompts import PromptTemplate

prompt_extract = PromptTemplate.from_template(
    """
    ### SCRAPED TEXT FROM TEMPLATE:
     {page_data}
     ### INSTRUCTION:
     The scraped text is from the careers page of a website.
     Your job is to extract the job postings and return them in JSON format containing only following keys: 'role', 'experience', 'skills' and 'description'.
     Only return the valid JSON.
     ### VALID JSON (NO PREAMBLE only JSON document):
    """
)
chain_extract = prompt_extract | llm
res = chain_extract.invoke(input={'page_data':page_data})
type(res.content)

str

In [23]:
from langchain_core.output_parsers import JsonOutputParser

json_parser = JsonOutputParser()
json_res = json_parser.parse(res.content)
json_res = json_res[0]

In [24]:
type(json_res)

dict

In [25]:
import pandas as pd
df = pd.read_csv('app/resource/my_portfolio.csv')
df.head()

,Techstack,Links
0,"React, Node.js, MongoDB",https://example.com/react-portfolio
1,"Angular,.NET, SQL Server",https://example.com/angular-portfolio
2,"Vue.js, Ruby on Rails, PostgreSQL",https://example.com/vue-portfolio
3,"Python, Django, MySQL",https://example.com/python-portfolio
4,"Java, Spring Boot, Oracle",https://example.com/java-portfolio


In [28]:
import chromadb
import uuid

client = chromadb.PersistentClient('vectorstore')
collection = client.get_or_create_collection(name = "portfolio")

if not collection.count():
    for _, row in df.iterrows():
        collection.add(documents=row["Techstack"],
                       metadatas={"links": row["Links"]},
                       ids = [str(uuid.uuid4())])

In [34]:
links = collection.query(query_texts=["Experience in Python"], n_results=2).get('metadatas')
links

[[{'links': 'https://example.com/ml-python-portfolio'},
  {'links': 'https://example.com/python-portfolio'}]]

In [32]:
job = json_res
job['skills']

['data structures',
 'algorithms',
 'software development',
 'programming languages',
 'infrastructure',
 'distributed systems',
 'networks',
 'compute technologies',
 'storage',
 'hardware architecture',
 'accessible technologies']

In [35]:
links = collection.query(query_texts=job['skills'], n_results=2).get('metadatas')
links

[[{'links': 'https://example.com/ios-portfolio'},
  {'links': 'https://example.com/magento-portfolio'}],
 [{'links': 'https://example.com/ml-python-portfolio'},
  {'links': 'https://example.com/android-portfolio'}],
 [{'links': 'https://example.com/ml-python-portfolio'},
  {'links': 'https://example.com/ios-ar-portfolio'}],
 [{'links': 'https://example.com/ml-python-portfolio'},
  {'links': 'https://example.com/magento-portfolio'}],
 [{'links': 'https://example.com/ml-python-portfolio'},
  {'links': 'https://example.com/devops-portfolio'}],
 [{'links': 'https://example.com/android-portfolio'},
  {'links': 'https://example.com/magento-portfolio'}],
 [{'links': 'https://example.com/ml-python-portfolio'},
  {'links': 'https://example.com/android-tv-portfolio'}],
 [{'links': 'https://example.com/ml-python-portfolio'},
  {'links': 'https://example.com/ios-ar-portfolio'}],
 [{'links': 'https://example.com/android-portfolio'},
  {'links': 'https://example.com/ios-portfolio'}],
 [{'links': 'ht

In [37]:
prompt_email = PromptTemplate.from_template(
        """
        ### JOB DESCRIPTION:
        {job_description}

        ### INSTRUCTION:
        You are Padmini, a business development executive at Titanic. Titanic is an AI & Software Consulting company dedicated to facilitating
        the seamless integration of business processes through automated tools.
        Over our experience, we have empowered numerous enterprises with tailored solutions, fostering scalability,
        process optimization, cost reduction, and heightened overall efficiency.
        Your job is to write a cold email to the Hiring Manager regarding the job mentioned above describing the capability of Titanic
        in fulfilling their needs.
        Also add the most relevant ones from the following links to showcase Titanic's portfolio: {link_list}
        Remember you are Padmini, BDE at Titanic.
        Do not provide a preamble.
        ### EMAIL (NO PREAMBLE):

        """
        )

chain_email = prompt_email | llm
res = chain_email.invoke({"job_description": str(job), "link_list": links})
print(res.content)

Subject: Expert Infrastructure Development Solutions for Your Organization

Dear Hiring Manager,

I came across the job posting for a Software Engineer III, Infrastructure, Core, and I'm excited to introduce Titanic, a leading AI & Software Consulting company, as a potential partner to fulfill your infrastructure development needs.

Our team of experts has extensive experience in designing and developing scalable, efficient, and secure infrastructure solutions. We possess in-depth knowledge of data structures, algorithms, software development, programming languages, infrastructure, distributed systems, networks, compute technologies, storage, hardware architecture, and accessible technologies.

We've empowered numerous enterprises with tailored solutions, fostering scalability, process optimization, cost reduction, and heightened overall efficiency. Our portfolio showcases our capabilities in:

* Developing scalable e-commerce platforms using Magento (https://example.com/magento-portfo